In [44]:
import sys
sys.path.append('utilities/')
import pandas as pd
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import torch
from sentence_transformers import SentenceTransformer
from joblib import dump
from openai import OpenAI
from tqdm import tqdm
from mmd import MMD
import re
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import ot
import random

In [21]:
sentence_transformer = SentenceTransformer('all-mpnet-base-v2')

In [190]:
old_train_url = 'data/generated/reddit/control/initial_gen.csv'
new_train_url = 'data/generated/reddit/control/new_init_gen.csv'
test_url = 'data/initial_datasets/reddit/reddit_test.csv'

In [191]:
train_df = pd.read_csv('data/initial_datasets/reddit/reddit_train.csv')

In [192]:
train_df = train_df.sample(n=1000, random_state=42)

In [193]:
old_train_df = pd.read_csv(old_train_url)
new_train_df = pd.read_csv(new_train_url)
test_df = pd.read_csv(test_url)

In [182]:
old_train_df

,messages,labels
0,I'm so ready for the new season of [NAME]'s sh...,0
1,"Ugh, I can't believe how bad the movie was, de...",1
2,"Just finished the book and omg it was amazing,...",0
3,"The weather today has been just awful, can't w...",1
4,Played the game with friends and we had such a...,0
...,...,...
995,"Missed the concert because of traffic, so bumm...",1
996,"My dog's new tricks are adorable, he's the bes...",0
997,Why is customer service so unhelpful? Worst ex...,1
998,Started reading this book and can't put it dow...,0


In [163]:
new_train_df

,messages,labels
0,I'm totally obsessed with this new series!! 🔥🔥,0
1,"Ugh, my computer crashed again... second time ...",1
2,"Best coffee shop in town, hands down! ☕️✨",0
3,"Can't believe I wasted money on this game, suc...",1
4,"Y'all, I just adopted the cutest puppy!! 🐶❤️",0
...,...,...
1995,Why is adulting so hard sometimes??? 😫,1
1996,"Cooked dinner for the fam, felt like a chef! 🍝",0
1997,Can't believe how rude some ppl can be on the ...,1
1998,"My dog learned a new trick today, so proud! 🐶",0


In [164]:
test_df

,messages,labels
0,First is the worst,1
1,Our education system has been a complete and u...,1
2,"The fuck you call me!? A cunt!? Damn man, didn...",1
3,It will probably take him some time to figure ...,0
4,Somebody is really insecure about their career...,1
...,...,...
995,Be glad you don't know the answer.,0
996,I’m laughing more that I feel I should st this...,0
997,I wouldnt necessarily call you and addict but ...,1
998,I'm genuinely interested in the responses to t...,0


In [209]:
train_df = pd.concat([new_train_df, old_train_df]).reset_index(drop=True)

In [210]:
train_df['labels'] = train_df['labels'].replace({-1: 1})

In [211]:
train_dfs = []
first_df = train_df.sample(n=1000, random_state=42)
rest_df = train_df.drop(first_df.index)
second_df = rest_df.sample(n=1000, random_state=42)
third_df = rest_df.drop(second_df.index)

In [212]:
train_dfs= [first_df, second_df, third_df]

In [176]:
from sklearn.metrics import confusion_matrix

In [213]:
results = {
    'Roc_Auc': [],
    'Prec': [],
    'Recall': [],
}

for i in range(3):
    train_df = train_dfs[i]
    
    train_df['labels'] = train_df['labels'].astype(int)
    test_df['labels'] = test_df['labels'].astype(int)
    X_train = train_df['messages']
    X_test = test_df['messages']
    
    y_train = train_df['labels']
    y_test = test_df['labels']
    print(train_df['labels'].value_counts())
    print(test_df['labels'].value_counts())
    X_train = np.array(sentence_transformer.encode(X_train.to_list()))
    X_test = np.array(sentence_transformer.encode(X_test.to_list()))

    model = svm.SVC(kernel='linear', probability=True, class_weight='balanced', random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_train)
    train_acc = accuracy_score(y_pred, y_train)

    y_pred = model.predict(X_test)
    test_acc = accuracy_score(y_pred, y_test)

    y_prob = model.predict_proba(X_test)[:, 1]
    print(y_test.iloc[:5])
    print(y_prob[:5])
    results['Roc_Auc'].append(roc_auc_score(y_test.to_list(), y_prob))
    results['Prec'].append(precision_score(y_test.to_list(), y_pred))
    results['Recall'].append(recall_score(y_test.to_list(), y_pred))

labels
1    503
0    497
Name: count, dtype: int64
labels
0    631
1    369
Name: count, dtype: int64
0    1
1    1
2    1
3    0
4    1
Name: labels, dtype: int64
[0.99999999 0.99999991 0.4841794  0.5173197  0.99999995]
labels
0    510
1    490
Name: count, dtype: int64
labels
0    631
1    369
Name: count, dtype: int64
0    1
1    1
2    1
3    0
4    1
Name: labels, dtype: int64
[0.99999998 1.         0.82491158 0.95176192 0.99999998]
labels
0    502
1    498
Name: count, dtype: int64
labels
0    631
1    369
Name: count, dtype: int64
0    1
1    1
2    1
3    0
4    1
Name: labels, dtype: int64
[0.99999842 0.9999992  0.71033797 0.19192563 0.99350893]


In [187]:
confusion_matrix(y_pred, y_test)

array([[364,  47],
       [267, 322]])

In [214]:
results

{'Roc_Auc': [np.float64(0.7917724264405878),
  np.float64(0.7813295882562629),
  np.float64(0.7851906252818472)],
 'Prec': [0.5255591054313099, 0.4881118881118881, 0.5477031802120141],
 'Recall': [0.8915989159891599, 0.94579945799458, 0.8401084010840109]}

In [174]:
for key in results:
    print(f"{key} mean: {np.mean(results[key])}")
    print(f"{key} std: {np.std(results[key], ddof=1)}")
    print()

Roc_Auc mean: 0.21501767315612932
Roc_Auc std: 0.006551599161238301

Prec mean: 0.5152919984831247
Prec std: 0.028378632839808588

Recall mean: 0.9024390243902438
Recall std: 0.04526637693357743



In [155]:
test_df = pd.read_csv('data/initial_datasets/reddit/reddit_test.csv')

In [141]:
test_df

,messages,labels
0,First is the worst,1
1,Our education system has been a complete and u...,1
2,"The fuck you call me!? A cunt!? Damn man, didn...",1
3,It will probably take him some time to figure ...,0
4,Somebody is really insecure about their career...,1
...,...,...
995,Be glad you don't know the answer.,0
996,I’m laughing more that I feel I should st this...,0
997,I wouldnt necessarily call you and addict but ...,1
998,I'm genuinely interested in the responses to t...,0


In [156]:
results = {
    'Roc_Auc': [],
    'Prec': [],
    'Recall': [],
}
seeds = [random.randint(0, 2**32 - 1) for _ in range(3)]
for i in range(3):
    gen_df = train_dfs[i]
    train_df = pd.concat([gen_df.sample(frac=0.9, random_state=seeds[i]), train_df.sample(frac=0.1, random_state=seeds[i])]).sample(frac=1, random_state=seeds[i]).reset_index(drop=True)
    test_df = test_df.sample(frac=1, random_state=seeds[i]).reset_index(drop=True)

    train_df['labels'] = train_df['labels'].astype(int)
    test_df['labels'] = test_df['labels'].astype(int)

    X_train = train_df['messages']
    X_test = test_df['messages']

    y_train = train_df['labels']
    y_test = test_df['labels']

    X_train = np.array(sentence_transformer.encode(X_train.to_list()))
    X_test = np.array(sentence_transformer.encode(X_test.to_list()))

    model = svm.SVC(kernel='linear', probability=True, class_weight='balanced', random_state=seeds[i])
    model.fit(X_train, y_train)

    y_pred = model.predict(X_train)
    train_acc = accuracy_score(y_pred, y_train)

    y_pred = model.predict(X_test)
    test_acc = accuracy_score(y_pred, y_test)

    y_prob = model.predict_proba(X_test)[:, 1]
    roc_auc_score(y_test, y_prob)

    results['Roc_Auc'].append(roc_auc_score(y_test, y_prob))
    results['Prec'].append(precision_score(y_test.to_list(), y_pred))
    results['Recall'].append(recall_score(y_test.to_list(), y_pred))

In [157]:
for key in results:
    print(f"{key} mean: {np.mean(results[key])}")
    print(f"{key} std: {np.std(results[key], ddof=1)}")
    print()

Roc_Auc mean: 0.8115772414987753
Roc_Auc std: 0.006668950937592346

Prec mean: 0.532216344195997
Prec std: 0.0036704376945273674

Recall mean: 0.8816621499548329
Recall std: 0.012807088418028712

